environment delpher

In [1]:
import os
import json
from lxml import etree
import requests

##### Handle single alto files

In [2]:
file_path = '../../../DATA/europeans-news/alto/urn=ddd_000023667_mpeg21_p003_alto.alto.xml'
filename = os.path.basename(file_path)
file_name = ''.join(os.path.splitext(filename)[0].split('.')[:-1])
file_name = file_name.replace('_', ':')
print(file_name)

urn=ddd:000023667:mpeg21:p003:alto


In [3]:
from gliner import GLiNER

labels = ["person", "location (geographical)", "organization"]

model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")
model.eval()
print("ok")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/traitlets

ok


In [4]:
def spans_to_bio(tokens, spans):
    # tokens: list[str]
    # spans: list[{"start": int, "end": int, "label": str}]

    token_dict = {}
    index = 0
    for i, token in enumerate(tokens):
        token_dict[i] = {"token": token, "offset": index, "label": "O"}
        index += len(list(token)) + 1

    # Update labels based on spans
    for span in spans:
        span_start = span["start"]
        span_end = span["end"]
        span_label = span["label"]
        
        first = True
        for token_idx in token_dict:
            token_offset = token_dict[token_idx]["offset"]
            # Check if token offset falls within span range
            if token_offset >= span_start and token_offset < span_end:
                if first:
                    token_dict[token_idx]["label"] = f"B-{span_label[:3].upper()}"
                    first = False
                else:
                    token_dict[token_idx]["label"] = f"I-{span_label[:3].upper()}"
    
    return token_dict

In [ ]:
# Function to split long text into chunks
# This is necessary because the model has a maximum input length, and we want to avoid truncation of the text.
def split_long_text(text, max_length=350):
    """Split text into chunks, add periods between chunks, and return updated text"""
    if len(text) <= max_length:
        return text
    
    words = text.split()
    chunks = []
    current_chunk = []
    current_length = 0
    
    for word in words:
        if current_length + len(word) + 1 > max_length and current_chunk:
            chunk_text = " ".join(current_chunk)
            # Add period if chunk doesn't end with one
            if not chunk_text.endswith('.'):
                chunk_text += '.'
            chunks.append(chunk_text)
            current_chunk = [word]
            current_length = len(word)
        else:
            current_chunk.append(word)
            current_length += len(word) + 1
    
    if current_chunk:
        chunk_text = " ".join(current_chunk)
        if not chunk_text.endswith('.'):
            chunk_text += '.'
        chunks.append(chunk_text)
    
    # Join chunks with space and return as single updated text
    return " ".join(chunks)

In [6]:
def create_block_of_text(page_tree):
    ns = {
            "alto": "http://schema.ccs-gmbh.com/ALTO"
        }
    block_of_text = []
    for text_block in page_tree.findall('.//alto:TextBlock', namespaces=ns):
        id = text_block.get("ID")
        text = []
        wc_score = []
        for string in text_block.findall('.//alto:String', namespaces=ns):
            text.append(string.get("CONTENT"))
            wc_score.append(float(string.get("WC")))
        block_of_text.append({
            "id": id,
            "text": " ".join(text),
            "wc_score": sum(wc_score) / len(wc_score) if wc_score else 0
        })
    return block_of_text

In [ ]:
blocks = create_block_of_text(page_tree)

with open(f"{file_name}.txt", "w") as f:
    f.close()

# Update block texts if they exceed max length
for block in blocks:
    if len(block["text"]) > 350:
        block["text"] = split_long_text(block["text"])

for block in blocks:

ID: P3_TB00001, Text: morgen zijn wij naar de mis geweest en zie hier hoe dezelve plaats had: «Geheel gewapend doch zonder randsel vergaderden wij ons op de markt, het muziekkorps van ongeveer 50 of 60 man aan het hoofd; wij gingen de kerk binnen en plaatsten ons aan weerszijden van het middenschip der kerk 4 man achter elkander, latende in het midden eene opening voor den bevelvoerenden kapitein ; achter het altaar zijn in eenen ronden vorm zetels geplaatst, waar de muzikanten plaats nemen; terstond na het Confiteor begint de muziek te spelen tot aan de konsekratie, de zouaven komen gewapend in de kerk, houden de pet op gedurende de geheele mis en brengen het geweer op schouder bij dea epistel, het evangelie en de nutting; gedurende de konse kratie presenteert men het geweer, buigt in die hou ding met aen regter knie op den grond en brengt de regter hand aan de pet : het is in die houding dat de zouaaf zijnen verlosser afwacht; daarna speelt de muziek tot aan het einde der mis, waar w

/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 595 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


muziekkorps => organization



ID: P3_TB00002, Text: In het No. van het Algemeen Muziekaal Tijdschrift van 15 Maart jl. leest men: TILBURG. Op 24 Januarij jl. gaf de liedertafel Souvenir des Montagnards haar jaarlijksch koncert, met medewerking van mej. van Boom, zangeres uit Parijs, den heer J. Dumon, onderwijzer aan het koninklijke conservatoire te Brussel en solo-fluitist van den koning der Belgen, en den heer Zurhaar, pianist uit Leipzig. Het programma was zamengesteld als volgt:, WC Score: 0.99
Algemeen Muziekaal Tijdschrift => organization
TILBURG => location (geographical)
Souvenir des Montagnards => organization
mej. van Boom => person
Parijs => location (geographical)
heer J. Dumon => person
koninklijke conservatoire te Brussel => organization
koning der Belgen => person
heer Zurhaar => person
Leipzig => location (geographical)



ID: P3_TB00003, Text: 1. Vlaggelied, (koor) van J. J. H. Verhulst, a. hocturne van F. Chopin; b. Galop, voor de piano forte van L. Brassin. Aria van

In [16]:
import pandas as pd
true_path = '../../../DATA/europeans-news/bio/'
pred_path = "."

true_df = pd.read_csv(os.path.join(true_path, f"{file_name}".replace(':', '_')+'.alto.xml.bio'), sep=' ', header=None, on_bad_lines='skip', engine='python')
pred_df = pd.read_csv(os.path.join(pred_path, f"{file_name}.txt"), sep='\t', header=None, on_bad_lines='skip', engine='python')

true_tags = [tag for tag in true_df.iloc[:, 2]]
pred_tags = [tag for tag in pred_df.iloc[:, 2]]

In [17]:
from seqeval.metrics import classification_report, f1_score


if len(true_tags) != len(pred_tags):
    print(f"Warning: true tags and predicted tags have different lengths ({len(true_tags)} vs {len(pred_tags)}). Truncating to the shorter length.")
    min_len = min(len(true_tags), len(pred_tags))
    true_tags = true_tags[:min_len]
    pred_tags = pred_tags[:min_len]

print(classification_report([true_tags], [pred_tags]))
print("\nF1 Score (micro avg):", f1_score([true_tags], [pred_tags]))

# Also print as dict for detailed metrics
report_dict = classification_report([true_tags], [pred_tags], output_dict=True)
print("\n" + "=" * 50)
print("DETAILED METRICS")
print("=" * 50)
for entity_type in sorted([k for k in report_dict.keys() if k not in ['micro avg', 'macro avg', 'weighted avg']]):
    metrics = report_dict[entity_type]
    print(f"{entity_type}:")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall: {metrics['recall']:.4f}")
    print(f"  F1-Score: {metrics['f1-score']:.4f}")

              precision    recall  f1-score   support

         LOC       0.87      0.49      0.62        41
         ORG       0.39      1.00      0.56         7
         PER       0.60      0.74      0.67        35

   micro avg       0.63      0.64      0.63        83
   macro avg       0.62      0.74      0.62        83
weighted avg       0.72      0.64      0.64        83


F1 Score (micro avg): 0.6347305389221557

DETAILED METRICS
LOC:
  Precision: 0.8696
  Recall: 0.4878
  F1-Score: 0.6250
ORG:
  Precision: 0.3889
  Recall: 1.0000
  F1-Score: 0.5600
PER:
  Precision: 0.6047
  Recall: 0.7429
  F1-Score: 0.6667


In [18]:
for t, p, true_token, pred_token in zip(true_tags, pred_tags, true_df.iloc[:, 0], pred_df.iloc[:, 0]):
    print(f"True: {t}, Pred: {p}, {true_token}, {pred_token}")

True: O, Pred: O, morgen, morgen
True: O, Pred: O, zijn, zijn
True: O, Pred: O, wij, wij
True: O, Pred: O, naar, naar
True: O, Pred: O, de, de
True: O, Pred: O, mis, mis
True: O, Pred: O, geweest, geweest
True: O, Pred: O, en, en
True: O, Pred: O, zie, zie
True: O, Pred: O, hier, hier
True: O, Pred: O, hoe, hoe
True: O, Pred: O, dezelve, dezelve
True: O, Pred: O, plaats, plaats
True: O, Pred: O, had:, had:
True: O, Pred: O, «Geheel, «Geheel
True: O, Pred: O, gewapend, gewapend
True: O, Pred: O, doch, doch
True: O, Pred: O, zonder, zonder
True: O, Pred: O, randsel, randsel
True: O, Pred: O, vergaderden, vergaderden
True: O, Pred: O, wij, wij
True: O, Pred: O, ons, ons
True: O, Pred: O, op, op
True: O, Pred: O, de, de
True: O, Pred: O, markt,, markt,
True: O, Pred: O, het, het
True: O, Pred: B-ORG, muziekkorps, muziekkorps
True: O, Pred: O, van, van
True: O, Pred: O, ongeveer, ongeveer
True: O, Pred: O, 50, 50
True: O, Pred: O, of, of
True: O, Pred: O, 60, 60
True: O, Pred: O, man, man
T